In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import torch
import numpy as np
import re
import dateparser

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from transformers import DistilBertTokenizerFast, DistilBertForTokenClassification
from transformers import Trainer, TrainingArguments

In [ ]:
df = pd.read_excel("/content/drive/MyDrive/Sem VI/Natural Language Processing/final_travel_dataset_7800.xlsx")

df = df.fillna("")
df = df.drop_duplicates(subset=["email_text"])

df["email_text"] = (
    df["email_text"]
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

df = df[['email_text','vehicle_number','from_location','to_location','date_expression','time']]

# TRAIN ONLY ON LABELED DATA
df_train = df[df['vehicle_number'] != ""]

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def create_ner_tags(text, entities):

    encoding = tokenizer(text, return_offsets_mapping=True)
    offsets = encoding["offset_mapping"]

    tags = ["O"] * len(offsets) # Initialize tags list

    for i, (start, end) in enumerate(offsets):
        if start == end:
            tags[i] = "O"
            continue

    for entity, label in entities:
        if entity == "":
            continue

        start_idx = text.lower().find(entity.lower())
        if start_idx == -1:
            continue

        end_idx = start_idx + len(entity)

        for i, (start, end) in enumerate(offsets):

            if start >= start_idx and end <= end_idx:
                if start == start_idx:
                    tags[i] = "B-" + label
                else:
                    tags[i] = "I-" + label

    return {
        "input_ids": encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels": tags
    }

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
data = []

for _, row in df_train.iterrows():

    entities = [
        (row['vehicle_number'], "MODE"),
        (row['from_location'], "FROM"),
        (row['to_location'], "TO"),
        (row['date_expression'], "DATE"),
        (row['time'], "TIME")
    ]

    # Modify to correctly unpack the dictionary returned by create_ner_tags
    processed_output = create_ner_tags(row['email_text'], entities)
    tags = processed_output['labels']

    data.append((row['email_text'], tags))

In [ ]:
label_list = [
"O",
"B-MODE","I-MODE",
"B-FROM","I-FROM",
"B-TO","I-TO",
"B-DATE","I-DATE",
"B-TIME","I-TIME"
]

label2id = {l:i for i,l in enumerate(label_list)}
id2label = {i:l for l,i in label2id.items()}

In [ ]:
train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)

In [ ]:
def tokenize_and_align(data):

    texts = [x[0] for x in data]
    tags = [x[1] for x in data]

    encodings = tokenizer(texts, padding=True, truncation=True, return_offsets_mapping=True)

    labels = []

    for i in range(len(tags)):

        word_ids = encodings.word_ids(batch_index=i)

        label_ids = []
        prev = None

        for word_id in word_ids:

            if word_id is None:
                label_ids.append(-100)

            elif word_id < len(tags[i]):
                label_ids.append(label2id[tags[i][word_id]])

            else:
                label_ids.append(-100)

        labels.append(label_ids)

    return encodings, labels

In [ ]:
class TravelDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k,v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
model = DistilBertForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    num_train_epochs=4
)

train_dataset = TravelDataset(*tokenize_and_align(train_data))
val_dataset = TravelDataset(*tokenize_and_align(val_data))

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

Step,Training Loss
500,0.554084
1000,0.216633
1500,0.132341
2000,0.088181


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2400, training_loss=0.217221519947052, metrics={'train_runtime': 176.7906, 'train_samples_per_second': 108.603, 'train_steps_per_second': 13.575, 'total_flos': 279316447372800.0, 'train_loss': 0.217221519947052, 'epoch': 4.0})

In [ ]:
predictions, labels, _ = trainer.predict(val_dataset)

preds = np.argmax(predictions, axis=2)

true_preds = []
true_labels = []

for pred, lab in zip(preds, labels):
    for p_, l_ in zip(pred, lab):
        if l_ != -100:
            true_preds.append(p_)
            true_labels.append(l_)

print("Accuracy:", accuracy_score(true_labels, true_preds))

unique_labels = sorted(set(true_labels))

print(classification_report(
    true_labels,
    true_preds,
    labels=unique_labels,
    target_names=[id2label[i] for i in unique_labels]
))

Accuracy: 0.9948677820993763
              precision    recall  f1-score   support

           O       1.00      1.00      1.00     25265
      B-MODE       0.99      0.98      0.99      1250
      I-MODE       0.99      1.00      1.00      4744
      B-FROM       0.99      0.99      0.99      1455
      I-FROM       0.99      0.99      0.99       836
        B-TO       0.99      0.99      0.99      1282
        I-TO       0.99      0.99      0.99       807
      B-DATE       0.98      0.98      0.98      1241
      I-DATE       0.99      1.00      0.99      2890
      B-TIME       1.00      0.98      0.99      1022
      I-TIME       0.99      1.00      0.99      2659

    accuracy                           0.99     43451
   macro avg       0.99      0.99      0.99     43451
weighted avg       0.99      0.99      0.99     43451



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def extract_entities(text):

    inputs = tokenizer(text, return_tensors="pt", truncation=True).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    preds = torch.argmax(outputs.logits, dim=2)[0]

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    entities = {}
    current_entity = ""
    current_label = None

    for token, pred in zip(tokens, preds):
        label = id2label[pred.item()]

        # ❌ Skip special tokens
        if token in ["[CLS]", "[SEP]", "[PAD]"]:
            continue

        # Fix subwords
        if token.startswith("##"):
            token = token[2:]
            current_entity += token
            continue

        # NEW ENTITY
        if label.startswith("B-"):
            if current_entity and current_label:
                entities.setdefault(current_label, []).append(current_entity.strip())

            current_entity = token
            current_label = label[2:]

        elif label.startswith("I-") and current_label == label[2:]:
            current_entity += " " + token

        else:
            if current_entity and current_label:
                entities.setdefault(current_label, []).append(current_entity.strip())

            current_entity = ""
            current_label = None

    if current_entity and current_label:
        entities.setdefault(current_label, []).append(current_entity.strip())

    return entities

In [ ]:
def validate_entities(entities):

    corrected = {}

    for key, values in entities.items():

        clean_vals = []

        for v in values:

            v = v.strip()

            # ❌ Remove obvious wrong predictions
            if len(v) < 3:
                continue

            # ❌ Remove time mistaken as date
            if key == "TO" and any(x in v.lower() for x in ["am","pm","monday","tuesday"]):
                continue

            # ❌ Remove numbers as location
            if key in ["FROM","TO"] and any(char.isdigit() for char in v):
                continue

            clean_vals.append(v)

        if clean_vals:
            corrected[key] = clean_vals

    return corrected

In [ ]:
# ================== VALIDATE REAL TRIP ==================
def is_valid_trip(entity):

    # At least 2 meaningful fields must exist
    keys = ["MODE", "FROM", "TO", "DATE", "TIME"]

    count = sum(1 for k in keys if k in entity)

    if count >= 2:
        return True

    return False

In [ ]:
# ================== EXTRACT BASE DATE ==================
# from datetime import datetime

# def extract_base_date(text):

#     # Try to find a real date in email
#     match = re.search(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}', text)

#     if match:
#         parsed = dateparser.parse(match.group())
#         if parsed:
#             return parsed

#     # fallback → today's system date
#     return datetime.now()
from datetime import datetime

def extract_base_date(text):

    # ❌ DO NOT pick travel dates from email
    # ✅ Use system current date instead

    return datetime.now()

In [ ]:
  def normalize_dates(entities, base_date):

    if "DATE" in entities:

        normalized = []

        for d in entities["DATE"]:

            parsed = dateparser.parse(
                d,
                settings={
                    'PREFER_DATES_FROM': 'future',
                    'RELATIVE_BASE': base_date
                }
            )

            if parsed:
                normalized.append(parsed.strftime("%d-%m-%Y"))

        if normalized:
            entities["DATE"] = normalized

    return entities

In [ ]:
# ================== RULE-BASED EXTRACTION ==================
def rule_based_extraction(text):

    result = {}

    # MODE (flight/train/bus)
    mode_match = re.findall(r'(Flight\s?[A-Z0-9]+|Train\s?\d+|Bus\s?[A-Z0-9]+)', text, re.I)
    if mode_match:
        result["MODE"] = mode_match


    # FROM-TO
    route_match = re.search(r'from\s+(\w+)\s+to\s+(\w+)', text, re.I)
    if route_match:
        result["FROM"] = [route_match.group(1)]
        result["TO"] = [route_match.group(2)]

    # TIME
    time_match = re.findall(r'\d{1,2}:\d{2}\s?(?:AM|PM)?', text, re.I)
    if time_match:
        result["TIME"] = time_match

    # DATE
    date_match = re.findall(r'(tomorrow|today|next \w+|\d{1,2}[/-]\d{1,2}[/-]\d{2,4})', text, re.I)
    if date_match:
        result["DATE"] = date_match

    return result

In [ ]:
# ================== UNIVERSAL EMAIL PROCESSOR ==================

def split_trips(text):
    # Split using ., newline, or "and then", "also"
    sentences = re.split(r'\.|\n| and | also ', text)
    return [s.strip() for s in sentences if s.strip()]

def process_email(text):

    base_date = extract_base_date(text)

    segments = re.split(r'\.|\n', text)
    trips = []

    for seg in segments:

        if not seg.strip():
            continue

        ner_entities = extract_entities(seg)
        ner_entities = clean_entities(ner_entities)

        rule_entities = rule_based_extraction(seg)

        final = {**ner_entities, **rule_entities}

        final = validate_entities(final)
        final = normalize_dates(final, base_date)

        if final and is_valid_trip(final):
          trips.append(final)

    # 🔥 GUARANTEED RETURN
    if len(trips) == 0:
        return {"message": "No travel info found"}

    elif len(trips) == 1:
        return {"type": "single_trip", "details": trips[0]}

    else:
        return {"type": "multi_trip", "trips": trips}

In [ ]:
def format_output(result):

    if result is None:
        print("❌ No output returned from model")
        return

    if "message" in result:
        print(result["message"])
        return

    if result["type"] == "single_trip":

        print("\n===== SINGLE TRIP =====\n")

        for key, value in result["details"].items():
            print(f"{key}: {', '.join(value)}")

    else:

        print(f"\n===== MULTI TRIP ({len(result['trips'])}) =====\n")

        for i, trip in enumerate(result["trips"], 1):

            print(f"--- Trip {i} ---")

            for key, value in trip.items():
                print(f"{key}: {', '.join(value)}")

            print()

In [ ]:
clean_entities = validate_entities

In [ ]:
email_text = """
Your journey starts with Flight AI202 from Delhi to Mumbai tomorrow at 10:30 AM.
Then Train 12951 from Mumbai to Pune next Monday at 6:00 AM.
Finally bus MH12AB1234 from Pune to Nagpur on 25/03/2026 at 8:45 PM.
"""

result = process_email(email_text)

format_output(result)


===== MULTI TRIP (3) =====

--- Trip 1 ---
MODE: Flight AI202
FROM: Delhi
TO: Mumbai
TIME: 10:30 AM
DATE: 03-04-2026

--- Trip 2 ---
MODE: Train 12951
FROM: Mumbai
DATE: next Monday
TO: Pune
TIME: 6:00 AM

--- Trip 3 ---
MODE: bus MH12AB1234
DATE: 25-03-2026
FROM: Pune
TO: Nagpur
TIME: 8:45 PM



In [ ]:
email_text = """
Hello Passenger,
Your flight 6E363 from Varanasi to Bhopal is scheduled for 10-02-2026 at 1:50 PM.
Please arrive early for boarding.
"""

result = process_email(email_text)

format_output(result)


===== SINGLE TRIP =====

MODE: flight 6E363
FROM: Varanasi
TO: Bhopal
TIME: 1:50 PM
DATE: 02-10-2026
